In [0]:
from pyspark.sql.functions import col, to_date, year, month, dayofmonth, dayofweek, avg, when

# --- 1. CARGAR DATOS BRONCE ---
# Leemos de bronze (esto no cambia)
df_city_day = spark.read.table("hive_metastore.default.bronze_city_day")
df_stations = spark.read.table("hive_metastore.default.bronze_stations")

# --- 2. CREAR DIM_TIEMPO ---
print("Construyendo dim_tiempo...")
df_tiempo = df_city_day.select("Date").distinct() \
    .withColumn("Fecha", to_date(col("Date"), "yyyy-MM-dd")) \
    .withColumn("Año", year(col("Fecha"))) \
    .withColumn("Mes", month(col("Fecha"))) \
    .withColumn("Dia", dayofmonth(col("Fecha"))) \
    .withColumn("DiaSemana", dayofweek(col("Fecha"))) \
    .withColumn("Trimestre", ((month(col("Fecha")) - 1) / 3 + 1).cast("int")) \
    .filter(col("Fecha").isNotNull())

# --- 3. CREAR DIM_CIUDAD ---
print("Construyendo dim_ciudad...")
df_ciudad = df_stations.select("City", "State").distinct() \
    .withColumnRenamed("City", "Ciudad") \
    .withColumnRenamed("State", "Estado") \
    .filter(col("Ciudad").isNotNull())

# --- 4. CREAR FACT_CALIDAD_AIRE ---
print("Construyendo fact_calidad_aire...")

# Usamos backticks (` `) para PM2.5
df_fact = df_city_day \
    .withColumn("Fecha", to_date(col("Date"), "yyyy-MM-dd")) \
    .withColumnRenamed("City", "Ciudad") \
    .select(
        "Ciudad", 
        "Fecha", 
        col("`PM2.5`").cast("double").alias("PM2_5"),
        col("PM10").cast("double").alias("PM10"),
        col("NO2").cast("double").alias("NO2"),
        col("SO2").cast("double").alias("SO2"),
        col("CO").cast("double").alias("CO"),
        col("O3").cast("double").alias("O3"),
        col("AQI").cast("double").alias("AQI"),
        col("AQI_Bucket")
    ) \
    .na.fill(0, ["PM2_5", "PM10", "AQI"]) 

# --- 5. GUARDAR EN PLATA (TABLAS DELTA) ---
# Aquí cambiamos el prefijo de 'silver_' a 'plata_'
df_tiempo.write.format("delta").mode("overwrite").saveAsTable("plata_dim_tiempo")
df_ciudad.write.format("delta").mode("overwrite").saveAsTable("plata_dim_ciudad")
df_fact.write.format("delta").mode("overwrite").saveAsTable("plata_fact_calidad_aire")

print("✅ Tablas Plata creadas EXITOSAMENTE: plata_dim_tiempo, plata_dim_ciudad, plata_fact_calidad_aire")

Construyendo dim_tiempo...
Construyendo dim_ciudad...
Construyendo fact_calidad_aire...
✅ Tablas Plata creadas EXITOSAMENTE: plata_dim_tiempo, plata_dim_ciudad, plata_fact_calidad_aire


In [0]:
# --- SCRIPT DE VALIDACIÓN DE CALIDAD DE DATOS (SILVER) ---

def verificar_tabla(nombre_tabla, columna_clave):
    print(f"--- 🔍 Auditando tabla: {nombre_tabla} ---")
    df = spark.read.table(nombre_tabla)
    
    # 1. Conteo de registros
    total = df.count()
    print(f"Total de registros: {total}")
    
    # 2. Verificar Nulos en Claves Primarias
    nulos = df.filter(col(columna_clave).isNull()).count()
    if nulos == 0:
        print(f"✅ Integridad de Clave ({columna_clave}): OK (0 nulos)")
    else:
        print(f"⚠️ ALERTA: Se encontraron {nulos} nulos en {columna_clave}")
        
    # 3. Muestra de datos
    print("Muestra de datos:")
    df.show(3, truncate=False)
    print("-" * 30)

# Ejecutar validaciones
verificar_tabla("silver_dim_tiempo", "Fecha")
verificar_tabla("silver_dim_ciudad", "Ciudad")
verificar_tabla("silver_fact_calidad_aire", "AQI")

# 4. Verificación de Integridad Referencial (Join de prueba)
print("--- 🔄 Prueba de Modelo Estrella (Join Fact + Dims) ---")
df_join = spark.read.table("silver_fact_calidad_aire").alias("f") \
    .join(spark.read.table("silver_dim_ciudad").alias("c"), col("f.Ciudad") == col("c.Ciudad"), "inner") \
    .join(spark.read.table("silver_dim_tiempo").alias("t"), col("f.Fecha") == col("t.Fecha"), "inner")

count_join = df_join.count()
print(f"Registros consolidados en el modelo estrella: {count_join}")

if count_join > 0:
    print("✅ El modelo estrella funciona correctamente. Las tablas cruzan bien.")
else:
    print("❌ ERROR: El join devolvió 0 filas. Revisa los nombres de ciudades o formatos de fecha.")

--- 🔍 Auditando tabla: silver_dim_tiempo ---
Total de registros: 2009
✅ Integridad de Clave (Fecha): OK (0 nulos)
Muestra de datos:
+----------+----------+----+---+---+---------+---------+
|Date      |Fecha     |Año |Mes|Dia|DiaSemana|Trimestre|
+----------+----------+----+---+---+---------+---------+
|2015-03-09|2015-03-09|2015|3  |9  |2        |1        |
|2015-05-19|2015-05-19|2015|5  |19 |3        |2        |
|2016-03-01|2016-03-01|2016|3  |1  |3        |1        |
+----------+----------+----+---+---+---------+---------+
only showing top 3 rows
------------------------------
--- 🔍 Auditando tabla: silver_dim_ciudad ---
Total de registros: 127
✅ Integridad de Clave (Ciudad): OK (0 nulos)
Muestra de datos:
+-----------------+--------------+
|Ciudad           |Estado        |
+-----------------+--------------+
|Ramanagara       |Karnataka     |
|Rajamahendravaram|Andhra Pradesh|
|Hajipur          |Bihar         |
+-----------------+--------------+
only showing top 3 rows
-------------